# 12c — Template Consistency Check across diaSources for a diaObject

## Purpose

Verify whether the **template cutouts** sent by Rubin via Fink alerts are **identical across
all diaSources** for a given `diaObjectId` and band, or whether they differ visit-to-visit.

Differences in templates could originate from:
- PSF-convolution of the template adapted to each visit's PSF (PSF-matched coadd templates)
- Small rotations / WCS differences between visits (pixel-grid resampling)
- Different template epochs if the deep coadd is updated during the survey
- Variable cutout sizes delivered by the alert stream

## Method

For each band independently:
1. Load template arrays for all diaSources, sorted chronologically.
2. Designate the **first visit's template** as the reference `T_ref`.
3. For each subsequent visit `k`, pad both `T_ref` and `T_k` to the same size **centred on
   their geometric centres** (NaN-padding at borders, then converted to zero for display).
4. Compute the residual `ΔT_k = T_k − T_ref` and display it in a subplot grid using a
   diverging colormap with shared symmetric colour scale `±vmax`.
5. The first panel always shows `T_ref` itself for reference.

## Layout per band

```
┌───────┬────────┬────────┬────────┬────────┐
│ T_ref │ ΔT_1   │ ΔT_2   │ ΔT_3   │  ...   │  ← one row per NCOLS
│(gray) │(RdBu_r)│(RdBu_r)│(RdBu_r)│        │
└───────┴────────┴────────┴────────┴────────┘
```

The title of each `ΔT_k` panel includes: visit id, date, template size, and
`max|ΔT|` as a proxy for the amplitude of the difference.

---
- **Author:** Sylvie Dagoret-Campagne — IJCLab / IN2P3 / CNRS — Université Paris-Saclay
- **Creation date:** 2026-05-15
- **Based on:** 12b_viewCutouts.ipynb
- **Subject:** Fink/LSST DIA — Template stability check


## 1. Imports & configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from astropy.time import Time

warnings.filterwarnings("ignore")
print(f"pandas {pd.__version__}  |  numpy {np.__version__}")

In [ ]:
%matplotlib inline

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# USER PARAMETERS  ← edit here
# ─────────────────────────────────────────────────────────────────────────────


objsid = {
    0: 170032917163016282,
    1: 170094456334188639,
    2: 170019717277810735,
    3: 170235219334922248,
    4: 170094456400773135,
    5: 170226393328648251,
    6: 170257199649521669,
    7: 170094456357257301,
}

DIAOBJECT_ID = objsid[1]


# Bands to check: None = all available, or e.g. ["r", "i"]
BANDS_FILTER = None

# Number of columns in the subplot grid per band (first slot = T_ref, rest = ΔT_k)
NCOLS = 5

# Percentile used for the symmetric colour scale of the difference panels
DIFF_PERCENTILE = 99.0

# Percentile stretch for the T_ref reference panel
REF_STRETCH_LO = 1.0
REF_STRETCH_HI = 99.0

# Colormaps
CMAP_REF = "gray"
CMAP_DIFF = "RdBu_r"  # diverging: red = positive excess, blue = deficit

# Save figures?
SAVE_FIGS = True
DIR_FIGS = "figs_FINK_BLOCK_LC_12c"

# ─────────────────────────────────────────────────────────────────────────────
# Fixed paths (same layout as 12b)
# ─────────────────────────────────────────────────────────────────────────────
DIR_CUTOUTS = f"fullcutouts_{DIAOBJECT_ID}"
FILE_MANIFEST = os.path.join(DIR_CUTOUTS, "manifest.csv")
FILE_GAIA_MATCHED = os.path.join("data_FINK_BLOCK_LC_08b", "fink_diaobj_gaia_join_matched.csv")

BAND_ORDER = list("ugrizy")
BAND_COLORS = {"u": "#9b59b6", "g": "#2ecc71", "r": "#e74c3c", "i": "#e67e22", "z": "#3498db", "y": "#795548"}

os.makedirs(DIR_FIGS, exist_ok=True)
print(f"diaObjectId : {DIAOBJECT_ID}")
print(f"Manifest    : {os.path.abspath(FILE_MANIFEST)}")
print(f"Figures dir : {os.path.abspath(DIR_FIGS)}")

## 2. Utility functions

In [ ]:
def zscale(arr, lo=1.0, hi=99.0):
    """Return (vmin, vmax) from percentile-based z-scale, ignoring NaNs."""
    finite = arr[np.isfinite(arr)]
    if len(finite) == 0:
        return 0.0, 1.0
    return float(np.percentile(finite, lo)), float(np.percentile(finite, hi))


def symvlim(arr, percentile=99.0):
    """Return the symmetric half-range for a diverging colour scale."""
    finite = arr[np.isfinite(arr)]
    if len(finite) == 0:
        return 1e-9
    return max(float(np.percentile(np.abs(finite), percentile)), 1e-9)


def load_npy(src_id, band, kind):
    """Load fullcutouts_{obj}/cutouts/{src_id}_{band}_{kind}.npy → float32 or None."""
    fpath = os.path.join(DIR_CUTOUTS, "cutouts", f"{src_id}_{band}_{kind}.npy")
    return np.load(fpath).astype(np.float32) if os.path.exists(fpath) else None


def parse_bool(val):
    """Coerce a dipole flag (bool / int / str / NaN) to Python bool."""
    if isinstance(val, bool):
        return val
    if isinstance(val, (int, float)) and np.isfinite(float(val)):
        return bool(int(val))
    if isinstance(val, str):
        return val.strip().lower() in ("true", "1", "yes")
    return False


def mjd_to_date(mjd):
    """MJD (TAI) → ISO date string YYYY-MM-DD."""
    try:
        return Time(float(mjd), format="mjd", scale="tai").isot[:10]
    except Exception:
        return "?"


def pad_to_common_size_centred(arr_a, arr_b):
    """Pad two 2-D arrays symmetrically so they share the same shape.

    Both arrays are embedded in a canvas whose size is
    ``(max(ny_a, ny_b), max(nx_a, nx_b))``.  The original pixel centres
    are mapped to the centre of the output canvas, so that
    pixel-level subtraction is geometrically meaningful when the alert
    cutouts are aligned on the source position (which Rubin guarantees
    by centring the stamp on the diaSource position).

    Padding value is NaN; after subtraction NaN pixels are converted to
    0 for display purposes.

    Parameters
    ----------
    arr_a, arr_b : np.ndarray (2-D, float32)

    Returns
    -------
    padded_a, padded_b : np.ndarray  — same shape, float32
    """
    ny_a, nx_a = arr_a.shape
    ny_b, nx_b = arr_b.shape
    ny_out = max(ny_a, ny_b)
    nx_out = max(nx_a, nx_b)

    def _embed(arr, ny_out, nx_out):
        ny, nx = arr.shape
        # offsets to centre the original array inside the canvas
        row0 = (ny_out - ny) // 2
        col0 = (nx_out - nx) // 2
        canvas = np.full((ny_out, nx_out), np.nan, dtype=np.float32)
        canvas[row0 : row0 + ny, col0 : col0 + nx] = arr
        return canvas

    return _embed(arr_a, ny_out, nx_out), _embed(arr_b, ny_out, nx_out)


def draw_crosshair(ax, shape):
    """Draw a centred crosshair given an image shape (ny, nx)."""
    ny, nx = shape
    ax.axvline(nx / 2, color="yellow", lw=0.8, ls="--", alpha=0.7)
    ax.axhline(ny / 2, color="yellow", lw=0.8, ls="--", alpha=0.7)


print("Utility functions defined.")

## 3. Load manifest

In [ ]:
if not os.path.exists(FILE_MANIFEST):
    raise FileNotFoundError(
        f"{FILE_MANIFEST} not found.\nRun: python fink_download_full_cutouts.py --obj_id {DIAOBJECT_ID}"
    )

df = pd.read_csv(FILE_MANIFEST)
df["isDipole"] = df["r:isDipole"].fillna(False).apply(parse_bool)
df = df.sort_values("r:midpointMjdTai").reset_index(drop=True)

print(f"Manifest loaded : {len(df)} diaSources")
print(f"Bands available : {sorted(df['r:band'].unique())}")

## 4. Load Gaia DR3 object-level metadata

In [ ]:
gaia_G_mag, gaia_status, fink_group, gaia_dr3_id = np.nan, "?", "?", "?"

if os.path.exists(FILE_GAIA_MATCHED):
    df_gaia = pd.read_csv(FILE_GAIA_MATCHED)
    hit = df_gaia[df_gaia["diaObjectId"].astype(str) == str(DIAOBJECT_ID)]
    if len(hit):
        r = hit.iloc[0]
        gaia_G_mag = float(r.get("gaia_phot_g_mean_mag", np.nan))
        gaia_status = str(r.get("gaia_status", "?"))
        fink_group = str(r.get("group", "?"))
        gaia_dr3_id = str(r.get("dr3_source_id", "?"))

G_str = f"{gaia_G_mag:.2f}" if np.isfinite(gaia_G_mag) else "?"
print(f"Gaia G={G_str}  status={gaia_status}  group={fink_group}  DR3={gaia_dr3_id}")

## 5. Template sizes inventory

Before plotting differences, print a summary of template stamp sizes per band
to immediately flag variability.

In [ ]:
bands_available = [b for b in BAND_ORDER if b in df["r:band"].values]
bands_to_check = [b for b in bands_available if BANDS_FILTER is None or b in BANDS_FILTER]

print(f"Bands to check : {bands_to_check}\n")
print(f"{'Band':>5}  {'visit':>18}  {'date':>12}  {'tmpl shape':>12}  {'sci shape':>12}")
print("-" * 70)

for band in bands_to_check:
    df_b = df[df["r:band"] == band].sort_values("r:midpointMjdTai").reset_index(drop=True)
    for _, row in df_b.iterrows():
        src_id = str(row["r:diaSourceId"])
        visit = row.get("r:visit", "?")
        date = mjd_to_date(float(row["r:midpointMjdTai"]))
        tmp = load_npy(src_id, band, "Template")
        sci = load_npy(src_id, band, "Science")
        ts = f"{tmp.shape[0]}×{tmp.shape[1]}" if tmp is not None else "MISSING"
        ss = f"{sci.shape[0]}×{sci.shape[1]}" if sci is not None else "MISSING"
        print(f"{band:>5}  {str(visit):>18}  {date:>12}  {ts:>12}  {ss:>12}")
    print()

## 6. Template difference grids — one figure per band

Each subplot grid has:
- **First panel** : `T_ref` (template from the first visit, in grey)
- **Subsequent panels** : `ΔT_k = T_k − T_ref` (diverging colourmap)

Both reference and difference images are padded to a common canvas so that
their geometrical centres coincide (see `pad_to_common_size_centred`).

In [ ]:
for band in bands_to_check:
    bcolor = BAND_COLORS.get(band, "gray")
    df_b = df[df["r:band"] == band].sort_values("r:midpointMjdTai").reset_index(drop=True)
    n_src = len(df_b)

    print(f"\n{'=' * 70}")
    print(f"  Band {band}  —  {n_src} diaSources")
    print(f"{'=' * 70}")

    # ── 1. Load all template arrays ──────────────────────────────────────
    templates = []  # list of (src_id, visit, date, shape_str, array_or_None)
    for _, row in df_b.iterrows():
        src_id = str(row["r:diaSourceId"])
        visit = row.get("r:visit", "?")
        date = mjd_to_date(float(row["r:midpointMjdTai"]))
        arr = load_npy(src_id, band, "Template")
        shape_str = f"{arr.shape[0]}×{arr.shape[1]}" if arr is not None else "MISSING"
        templates.append((src_id, visit, date, shape_str, arr))

    # Filter to entries that have a valid template
    valid = [(sid, v, d, ss, a) for sid, v, d, ss, a in templates if a is not None]
    if len(valid) == 0:
        print(f"  No template arrays found for band {band} — skipping.")
        continue

    # Reference = first valid template
    src_id_ref, visit_ref, date_ref, shape_ref, T_ref = valid[0]
    print(f"  Reference template : visit={visit_ref}  date={date_ref}  shape={shape_ref}")

    # ── 2. Build list of panels: [T_ref display] + [ΔT_k for k=1..N-1] ──
    # Panel 0: T_ref itself (just displayed in grey)
    # Panels 1..N-1: difference T_k - T_ref  (diverging)
    panels = []
    # Panel 0 — reference
    panels.append(
        {
            "kind": "ref",
            "label": f"T_ref\nvisit {visit_ref}\n{date_ref}\n{shape_ref}",
            "arr": T_ref,
            "max_abs_diff": None,
        }
    )

    for k, (sid_k, visit_k, date_k, shape_k, T_k) in enumerate(valid[1:], start=1):
        # Pad both arrays to a common size, centred
        T_ref_pad, T_k_pad = pad_to_common_size_centred(T_ref, T_k)

        # Difference — NaN where either side is padded; set to 0 for display
        delta = T_k_pad - T_ref_pad
        # Compute max absolute difference on valid (non-NaN) pixels
        valid_mask = np.isfinite(delta)
        max_abs = float(np.nanmax(np.abs(delta[valid_mask]))) if valid_mask.any() else 0.0
        # Replace NaN with 0 for display
        delta_disp = np.where(np.isfinite(delta), delta, 0.0)

        out_ny, out_nx = T_ref_pad.shape
        panels.append(
            {
                "kind": "diff",
                "label": f"ΔT_{k}  (T_{k} − T_ref)\nvisit {visit_k}\n{date_k}\nT_ref:{shape_ref} / T_{k}:{shape_k}",
                "arr": delta_disp,
                "max_abs_diff": max_abs,
                "canvas_shape": (out_ny, out_nx),
            }
        )
        print(
            f"  ΔT_{k:02d} : visit={visit_k}  date={date_k}  "
            f"shapes {shape_ref} vs {shape_k}  max|ΔT|={max_abs:.2f}"
        )

    # ── 3. Global shared vmax for all difference panels ──────────────────
    diff_arrays = [p["arr"] for p in panels if p["kind"] == "diff"]
    if diff_arrays:
        combined_diff = np.concatenate([a.ravel() for a in diff_arrays])
        vmax_diff = symvlim(combined_diff, percentile=DIFF_PERCENTILE)
    else:
        vmax_diff = 1.0
    print(f"  Shared diff colour scale : ±{vmax_diff:.2f}")

    # Colour scale for T_ref panel
    vmin_ref, vmax_ref = zscale(T_ref, REF_STRETCH_LO, REF_STRETCH_HI)

    # ── 4. Build figure ──────────────────────────────────────────────────
    n_panels = len(panels)
    # Distribute panels into rows of NCOLS
    n_cols = min(NCOLS, n_panels)
    n_rows = int(np.ceil(n_panels / n_cols))

    fig_width = 3.5 * n_cols
    fig_height = 3.8 * n_rows
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(fig_width, fig_height),
        squeeze=False,
        gridspec_kw={"hspace": 0.55, "wspace": 0.35},
    )

    for idx, panel in enumerate(panels):
        row_i = idx // n_cols
        col_i = idx % n_cols
        ax = axes[row_i][col_i]

        arr = panel["arr"]

        if panel["kind"] == "ref":
            im = ax.imshow(arr, origin="lower", cmap=CMAP_REF, vmin=vmin_ref, vmax=vmax_ref)
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            draw_crosshair(ax, arr.shape)
            ax.set_title(panel["label"], fontsize=7.5, color="black")
        else:
            im = ax.imshow(arr, origin="lower", cmap=CMAP_DIFF, vmin=-vmax_diff, vmax=vmax_diff)
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            draw_crosshair(ax, arr.shape)
            # Annotate max|ΔT| in the corner
            ax.text(
                0.03,
                0.97,
                f"max|ΔT|={panel['max_abs_diff']:.2f}",
                transform=ax.transAxes,
                fontsize=7,
                va="top",
                ha="left",
                color="white",
                bbox=dict(boxstyle="round,pad=0.2", fc="black", alpha=0.55),
            )
            ax.set_title(panel["label"], fontsize=7.5, color="#333")

        ax.axis("off")

    # Hide unused axes
    for idx in range(n_panels, n_rows * n_cols):
        axes[idx // n_cols][idx % n_cols].axis("off")

    fig.suptitle(
        f"Template consistency check — diaObjectId={DIAOBJECT_ID}   band={band}\n"
        f"Gaia G={G_str}  status={gaia_status}  group={fink_group}\n"
        f"Ref template: visit={visit_ref}  {date_ref}  shape={shape_ref}   "
        f"  Colour scale diff: ±{vmax_diff:.2f} (p{DIFF_PERCENTILE:.0f})",
        fontsize=10,
        color=bcolor,
        fontweight="bold",
        y=1.01,
    )
    plt.tight_layout()

    if SAVE_FIGS:
        stem = f"tmpl_check_obj{DIAOBJECT_ID}_band{band}"
        for ext in ("pdf", "png"):
            plt.savefig(
                os.path.join(DIR_FIGS, f"{stem}.{ext}"),
                bbox_inches="tight",
                dpi=120,
            )
        print(f"  → saved {stem}.{{pdf,png}}")

    plt.show()
    plt.close(fig)

## 7. Quantitative summary — max|ΔT| and pixel-level statistics

Print a compact table of residual statistics (mean, std, max|ΔT|) for every
band × visit pair.  Also show the relative amplitude `max|ΔT| / median(T_ref)`
as a rough measure of how significant the template variation is.

In [ ]:
from IPython.display import display as ipy_display

records = []

for band in bands_to_check:
    df_b = df[df["r:band"] == band].sort_values("r:midpointMjdTai").reset_index(drop=True)

    valid = []
    for _, row in df_b.iterrows():
        src_id = str(row["r:diaSourceId"])
        visit = row.get("r:visit", "?")
        date = mjd_to_date(float(row["r:midpointMjdTai"]))
        arr = load_npy(src_id, band, "Template")
        if arr is not None:
            valid.append((src_id, visit, date, arr))

    if len(valid) < 2:
        continue

    src_id_ref, visit_ref, date_ref, T_ref = valid[0]
    median_ref = float(np.nanmedian(T_ref))

    for k, (sid_k, visit_k, date_k, T_k) in enumerate(valid[1:], start=1):
        T_ref_pad, T_k_pad = pad_to_common_size_centred(T_ref, T_k)
        delta = T_k_pad - T_ref_pad
        valid_pix = delta[np.isfinite(delta)]

        mean_delta = float(np.mean(valid_pix)) if len(valid_pix) else np.nan
        std_delta = float(np.std(valid_pix)) if len(valid_pix) else np.nan
        max_abs = float(np.max(np.abs(valid_pix))) if len(valid_pix) else np.nan
        rel_max = max_abs / abs(median_ref) if (median_ref != 0 and np.isfinite(median_ref)) else np.nan
        n_pix_tot = T_ref_pad.size
        n_pix_valid = int(np.sum(np.isfinite(delta)))
        shape_ref_s = f"{T_ref.shape[0]}×{T_ref.shape[1]}"
        shape_k_s = f"{T_k.shape[0]}×{T_k.shape[1]}"

        records.append(
            {
                "band": band,
                "visit_ref": visit_ref,
                "date_ref": date_ref,
                "shape_ref": shape_ref_s,
                "visit_k": visit_k,
                "date_k": date_k,
                "shape_k": shape_k_s,
                "n_pix_valid": n_pix_valid,
                "n_pix_canvas": n_pix_tot,
                "mean_delta": round(mean_delta, 4),
                "std_delta": round(std_delta, 4),
                "max_abs_delta": round(max_abs, 4),
                "median_T_ref": round(median_ref, 4),
                "rel_max": round(rel_max, 4) if np.isfinite(rel_max) else np.nan,
            }
        )

if records:
    df_stats = pd.DataFrame(records)
    print(f"Template residual statistics for diaObjectId={DIAOBJECT_ID}")
    ipy_display(df_stats)

    # Save to CSV
    csv_path = os.path.join(DIR_FIGS, f"tmpl_check_stats_obj{DIAOBJECT_ID}.csv")
    df_stats.to_csv(csv_path, index=False)
    print(f"\n→ Statistics saved to {csv_path}")
else:
    print("Not enough valid template arrays for statistics (need ≥ 2 per band).")

## 8. Cross-correlation peak offset analysis

If templates differ by a sub-pixel shift or rotation, the cross-correlation
peak between `T_k` and `T_ref` will be offset from (0,0).
This cell measures that offset via `numpy.fft` for every valid pair and
prints a summary table.

In [ ]:
def xcorr_offset(arr_a, arr_b):
    """Estimate the translational offset of arr_b relative to arr_a via FFT cross-correlation.

    NaN pixels are replaced by the local median before FFT.

    Returns
    -------
    dy, dx : float
        Sub-pixel shift estimate (row, col) — positive means arr_b is shifted
        down/right relative to arr_a.
    peak_snr : float
        Peak value of the normalised cross-correlation (1.0 = perfect match).
    """

    def _clean(a):
        out = a.copy().astype(np.float64)
        med = np.nanmedian(out)
        out[~np.isfinite(out)] = med
        return out - np.mean(out)  # zero-mean for cleaner FFT

    # Pad to common size
    a_pad, b_pad = pad_to_common_size_centred(arr_a.astype(np.float32), arr_b.astype(np.float32))
    A = _clean(a_pad)
    B = _clean(b_pad)

    # Normalised cross-correlation via FFT
    FA = np.fft.fft2(A)
    FB = np.fft.fft2(B)
    denom = np.abs(FA) * np.abs(FB)
    denom[denom == 0] = 1.0
    xcorr = np.fft.ifft2(FA * np.conj(FB) / denom).real

    # Peak location (unwrapped to ±N/2)
    peak_idx = np.unravel_index(np.argmax(xcorr), xcorr.shape)
    ny, nx = xcorr.shape
    dy = int(peak_idx[0]) if int(peak_idx[0]) <= ny // 2 else int(peak_idx[0]) - ny
    dx = int(peak_idx[1]) if int(peak_idx[1]) <= nx // 2 else int(peak_idx[1]) - nx
    peak_snr = float(xcorr.max()) / (float(xcorr.std()) + 1e-30)

    return float(dy), float(dx), float(peak_snr)


xcorr_records = []

for band in bands_to_check:
    df_b = df[df["r:band"] == band].sort_values("r:midpointMjdTai").reset_index(drop=True)

    valid = []
    for _, row in df_b.iterrows():
        src_id = str(row["r:diaSourceId"])
        visit = row.get("r:visit", "?")
        date = mjd_to_date(float(row["r:midpointMjdTai"]))
        arr = load_npy(src_id, band, "Template")
        if arr is not None:
            valid.append((src_id, visit, date, arr))

    if len(valid) < 2:
        continue

    _, visit_ref, date_ref, T_ref = valid[0]

    for k, (sid_k, visit_k, date_k, T_k) in enumerate(valid[1:], start=1):
        dy, dx, snr = xcorr_offset(T_ref, T_k)
        xcorr_records.append(
            {
                "band": band,
                "visit_ref": visit_ref,
                "visit_k": visit_k,
                "date_k": date_k,
                "dy_pix": dy,
                "dx_pix": dx,
                "offset_pix": round(np.hypot(dy, dx), 3),
                "xcorr_snr": round(snr, 2),
            }
        )

if xcorr_records:
    df_xcorr = pd.DataFrame(xcorr_records)
    print("Cross-correlation offset of T_k relative to T_ref (pixels):")
    print("  offset_pix ~ 0 : identical grid orientation")
    print("  offset_pix > 0 : shift or rotation between templates")
    print()
    ipy_display(df_xcorr)

    csv_path = os.path.join(DIR_FIGS, f"tmpl_xcorr_obj{DIAOBJECT_ID}.csv")
    df_xcorr.to_csv(csv_path, index=False)
    print(f"\n→ Cross-correlation results saved to {csv_path}")
else:
    print("Not enough valid template arrays for cross-correlation (need ≥ 2 per band).")